# Packages import

In [80]:
import requests
from bs4 import BeautifulSoup
import json
import os 

# Ceneo Scraper
1.Provide ulr address of product's opinions webpage

In [81]:

product_id = 124893467
page = 1
url = f"https://www.ceneo.pl/{product_id}/opinie-{page}"
print(url)


https://www.ceneo.pl/124893467/opinie-1


2.Send the request to provided url adress

In [82]:
response = requests.get(url=url)

3. If status code is Ok, fetch all opinions from requested webpage

In [83]:
page_dom = BeautifulSoup(response.text,'html.parser')
soup = page_dom.find('h1',{'class':'product-top__product-info__name'}).get_text()

In [84]:
# print(page_dom.find_all('div',{'class':'user-post__card'}))
all_opinions = page_dom.find_all('div',{'class':'js_product-review'})


In [85]:
opinions = [opinion for opinion in page_dom.find_all('div',{'class':'js_product-review'}) if "user-post--highlight" not in  opinion.get("class",[])]
print(len(opinions))
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(len(opinions))

10
10


5. For all fetched opinions parse them to extract relevant data

In [86]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        "opinion_id":opinion["data-entry-id"],
        "author":opinion.select_one("span.user-post__author-name").get_text().strip(),
        'reccomendation':opinion.select_one('span.user-post__author-recommendation>em').get_text().strip() if opinion.
            select_one('span.user-post__author-recommendation>em') else None,
        "score":opinion.select_one('.user-post__score-count').get_text().strip(),
        "content":opinion.select_one('div.user-post__text').get_text().strip(),
        "pros":[p.get_text().strip() for p in opinion.select('div.review-feature__item--positive')],
        "cons":[c.get_text().strip() for c in opinion.select('div.review-feature__item--negative')],
        "like":opinion.select_one('button.vote-yes > span').get_text().strip(),
        "dislike":opinion.select_one('button.vote-no > span').get_text().strip(),
        "publishing_date":opinion.select_one('span.user-post__published > time:nth-child(1)')["datetime"].strip() if opinion.select_one('span.user-post__published > time:nth-child(1)') else None,
        "purchase_date":opinion.select_one('span.user-post__published > time:nth-child(2)')['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)') else None,
    }
    all_opinions.append(single_opinion)
    
json.dump(all_opinions,open("all_opinions.json",'w'),indent=4)


6 Check if there is next page


In [87]:
next = True if page_dom.select_one("button.pagination__next") else False
if next: page+=1


8. Save obtained opinion


In [88]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [89]:
with open(f"./opinions/{product_id}.json",'w',encoding="UTF-8") as file:
    json.dump(all_opinions,file,indent=4)